# 10.1 一張紅色方塊，怎麼存成圖片 tensor？


看一張黑底紅方塊：中央像素是紅色，左上角是黑色。希望程式讀出中央的 `[1,0,0]` 與左上的 `[0,0,0]`，還要說清楚這些數字屬於哪一張圖、哪個位置。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-10-pixels.svg")))

RGB 是紅、綠、藍三個通道。一個像素的三個亮度合在一起才是它的顏色；本例 0 是沒有該色亮度，1 是生成器設定的最大亮度。tensor 是帶形狀的數字陣列，圖片用 `[C,H,W]` 排列：通道、高、寬。所以 `[3,16,16]` 表示一張圖的三種顏色通道，不是三張圖。


In [ ]:
from tiny_perceptron.multimodal import scene

image = scene("red", "square")
print("形狀", tuple(image.shape))
print("中央RGB", image[:, 8, 8].tolist())
print("左上RGB", image[:, 0, 0].tolist())
print("加入批次軸", tuple(image.unsqueeze(0).shape))

`scene("red","square")` 直接生成 16×16 的黑底紅方塊。`image[:,8,8]` 保留全部通道，取第 8 列、第 8 欄；索引從 0 開始。前三行印出 `(3,16,16)`、`[1.0,0.0,0.0]`、`[0.0,0.0,0.0]`。`.tolist()` 只是將選出的數值轉成清單，沒有更動圖片。

`unsqueeze(0)` 增加批次軸，得到 `(1,3,16,16)`；第一個 1 表示一次有一張圖。增加軸不會新增像素。若讀檔工具給的是 `[H,W,C]`，需要交換軸，不能只改形狀名稱而讓內容排錯。

照片常用整數 0–255 保存亮度，轉成浮點數後可以除以 255。這裡的合成圖已經是 0–1，不要再除一次。通道順序與亮度尺度都要和後續模型的約定一致。

練習把 red 改成 blue：中央應變成 `[0,0,1]`，軸長度不變。變的是藍通道的亮度，並不是增加了第三張圖。

<details>
<summary>回顧與查證</summary>

可回顧：[W.3的tensor與軸](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.3)。

</details>
